In [1]:
!pip install fake

In [2]:
import pandas as pd 
import numpy as np 
from faker import Faker

fake = Faker()
np.random.seed(42)

df = pd.read_csv("../data/raw_sales.csv.csv")
print(df.shape)

(4999, 20)


In [3]:
products = df["product_id"].unique()
supplier_lookup = {
    p:{
        "supplier_id": f"SUP-{np.random.randint(100,199)}",
        "supplier_name": fake.company(),
        "lead_time_days":np.random.randint(3,21),
    }
    for p in products
}

sup_df = pd.DataFrame.from_dict(supplier_lookup, orient="index").reset_index()
sup_df.rename(columns= {"index":"product_id"},inplace=True)
df = df.merge(sup_df, on="product_id", how="left")

def get_category(row):
    for cat in ["Cabinets","Chairs","Sofas","Tables"]:
        if row [f"category_{cat}"]:
            return cat
    return "Unknown"

df["category"] = df.apply(get_category, axis=1)

df["region"] = df.apply(lambda r:"Europe" if r["region_Europe"] else"North America",axis=1)
df["store_type"] = df.apply(lambda r: "Retail" if r["store_type_Retail"] else "Wholesale",axis=1)

df.to_csv("../data/sales_with_supply_data.csv",index=False)
sup_df.to_csv("../data/supplier_reference.csv",index=False)

print("Done. Rows:",len(df))
print (df[["product_id","category","region","store_type","supplier_name","lead_time_days"]].
    head())
                            
                        

Done. Rows: 4999
   product_id  category         region store_type              supplier_name  \
0         151    Chairs  North America     Retail                Smith Group   
1         192  Cabinets  North America  Wholesale                 Turner Ltd   
2         114  Cabinets  North America  Wholesale               Bailey Group   
3         171    Chairs  North America  Wholesale  Martinez, Fields and Mora   
4         160     Sofas         Europe  Wholesale               Scott-Forbes   

   lead_time_days  
0              17  
1               9  
2              13  
3               6  
4               5  


In [1]:
import os
print(os.listdir("../data"))

['.ipynb_checkpoints', 'raw_sales.csv.csv', 'sales_with_supply_data.csv', 'supplier_reference.csv']
